# EHM Fleet Analytics — Phase 2: EDA
### Beginner-friendly version with full explanations
**Author:** Shami | **Platform:** Google Colab

---

## What this notebook does

We have a dataset of **20 engines × 3000 flight cycles each = 60,000 records**.  
Each row is one flight, with all the engine readings captured at cruise.

Our job as an analyst is to answer **five questions**:

| # | Question |
|---|---|
| Q1 | Which engines are behaving abnormally? |
| Q2 | How do I classify every engine into Red / Amber / Green? |
| Q3 | For the concerning engines — how long until they need the shop? |
| Q4 | Are the sensors working properly? |
| Q5 | Do short-haul engines degrade faster than long-haul ones? |

---

## How to run this notebook

1. Click on a cell
2. Press **Shift + Enter** to run it
3. Wait for it to finish (you'll see a green tick)
4. Move to the next cell, repeat

Run the cells **in order from top to bottom**. Don't skip.

---

## Section 0 — Getting the tools ready

Before we can do any analysis, we need to load Python's data tools.

Think of this like opening Excel before working on a spreadsheet — we need to bring in the software libraries we'll use.

**The libraries we're using:**
- **pandas** — for working with tables of data (like Excel in Python)
- **numpy** — for doing math on numbers
- **matplotlib** — for drawing charts
- **seaborn** — for making prettier charts

In [ ]:
# The 'import' command loads a library into our notebook so we can use it
# 'as pd' is just a nickname — instead of typing 'pandas' every time, we type 'pd'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# This line just hides warning messages that we don't need to see
import warnings
warnings.filterwarnings('ignore')

# Set up colours we'll use throughout — RAG stands for Red/Amber/Green
RED   = '#ef5350'   # Red for urgent
AMBER = '#ffa726'   # Amber for watch
GREEN = '#66bb6a'   # Green for healthy
BLUE  = '#42a5f5'   # Blue for general use

# Make our charts look clean with a dark background
plt.style.use('dark_background')
plt.rcParams['figure.facecolor']  = '#0f1117'
plt.rcParams['axes.facecolor']    = '#1a1d2e'
plt.rcParams['grid.alpha']        = 0.3

print('All libraries loaded. Ready to analyse data.')

## Section 1 — Upload the CSV file

We need to get your dataset into the notebook.

When you run the next cell, Colab will show an **upload button**. Click it and select your file:  
`ehm_synthetic_fleet.csv`

The file will be copied into Colab for this session.

In [ ]:
# This brings in Colab's file upload feature
from google.colab import files

# This line triggers the upload button — click it and pick your CSV
uploaded = files.upload()

In [ ]:
# Now that the file is uploaded, let's open it and look inside
# 
# 'pd.read_csv()' reads a CSV file and puts it into something called a
# DataFrame (essentially a table with rows and columns — like Excel)
# 
# We store that table in a variable called 'df' (short for 'dataframe')

df = pd.read_csv('ehm_synthetic_fleet.csv')

# Print some basic facts about the data so we know what we're working with
# The 'f' before the string lets us plug in variables using {curly braces}
print(f'Number of rows:     {len(df):,}')
print(f'Number of columns:  {len(df.columns)}')
print(f'Number of engines:  {df["engine_id"].nunique()}')
print()
print('First 5 rows:')
df.head()

### What just happened?

We loaded 60,000 rows of data into a table called `df`. Each row is one flight.  
The `.head()` command shows the first 5 rows so you can see the structure.

**Key columns to know:**
- `engine_id` — which engine (ENG-001 to ENG-020)
- `cycle` — flight number for that engine
- `EGT_margin` — the health metric we care about most
- `RUL` — remaining useful life in cycles
- `route_type` — short_haul / medium_haul / long_haul

## Section 2 — Build a fleet summary

Right now we have one row per flight (60,000 rows total).  
For a fleet manager's dashboard, we need **one row per engine** showing its current status.

Think of it like this: we have a log of every flight ever taken, but we want a summary that says "Engine 1 has flown X cycles, its margin is currently Y, etc."

We'll do this step by step.

In [ ]:
# STEP 1: Sort the data by cycle number (oldest flight first, latest flight last)
# This makes sure when we pick the 'latest' value, we really get the latest one

df = df.sort_values('cycle')

print('Data sorted by cycle.')

In [ ]:
# STEP 2: For each engine, pick out the LAST row
# This gives us the most recent snapshot of every engine
# 
# 'groupby' splits the data into groups (one per engine)
# '.last()' takes the final row from each group
# '.reset_index()' just cleans up the result into a normal table

latest = df.groupby('engine_id').last().reset_index()

print(f'Latest snapshot per engine: {len(latest)} rows (one per engine)')
latest[['engine_id', 'cycle', 'EGT_margin', 'RUL', 'route_type']].head()

In [ ]:
# STEP 3: For each engine, also get the FIRST row (its starting state)
# We need this to calculate how much the engine has degraded over its life

first = df.groupby('engine_id').first().reset_index()

# Now let's build a fleet summary table step by step
# We start with the latest snapshot and add columns we care about

fleet = latest[['engine_id', 'route_type', 'EGT_margin', 'RUL',
                'HPT_degradation', 'HPC_degradation',
                'SFC_degradation_pct']].copy()

# Rename 'EGT_margin' to 'EGT_margin_current' to make it clearer
fleet = fleet.rename(columns={
    'EGT_margin':  'EGT_margin_current',
    'RUL':         'RUL_current',
})

# Add the starting margin from the 'first' table
fleet['EGT_margin_start'] = first['EGT_margin'].values

# Calculate how many cycles each engine has flown (latest - first cycle number)
fleet['total_cycles'] = latest['cycle'].values - first['cycle'].values

print(f'Fleet summary table: {len(fleet)} engines')
fleet.head()

In [ ]:
# STEP 4: Calculate each engine's degradation RATE
# 
# How fast is this engine losing EGT margin?
# Rate = (starting margin - current margin) / total cycles × 100
# 
# We multiply by 100 so the number is 'margin lost per 100 cycles'
# which is easier to read than per 1 cycle

fleet['margin_lost']         = fleet['EGT_margin_start'] - fleet['EGT_margin_current']
fleet['deg_rate_per_100cyc'] = (fleet['margin_lost'] / fleet['total_cycles']) * 100

# Round to 2 decimal places for cleaner display
fleet['deg_rate_per_100cyc'] = fleet['deg_rate_per_100cyc'].round(2)

# Also count how many anomaly events each engine had
# We do this by summing the 'is_anomaly' column (where 1 = anomaly, 0 = normal)
anomaly_counts = df.groupby('engine_id')['is_anomaly'].sum().reset_index()
anomaly_counts.columns = ['engine_id', 'anomaly_count']

# Join the anomaly counts into the fleet table
fleet = fleet.merge(anomaly_counts, on='engine_id')

print('Fleet summary with degradation rates:')
fleet[['engine_id', 'route_type', 'EGT_margin_current',
       'RUL_current', 'deg_rate_per_100cyc', 'anomaly_count']].head()

In [ ]:
# STEP 5: Assign RAG status (Red / Amber / Green) to each engine
# 
# We'll use these rules (based on Phase 0 domain knowledge):
#   EGT margin below 30°C  → RED     (needs action now)
#   EGT margin 30-50°C     → AMBER   (plan maintenance)
#   EGT margin above 50°C  → GREEN   (healthy)
#
# Also: if RUL is low, we override to a worse colour

# First, set thresholds as variables so they're easy to change
MARGIN_RED_LIMIT   = 30    # Below this in °C = red
MARGIN_AMBER_LIMIT = 50    # Below this in °C = amber
RUL_RED_LIMIT      = 200   # Below this many cycles = red
RUL_AMBER_LIMIT    = 500   # Below this many cycles = amber

# Now we write a small function that takes ONE engine row and returns its colour
# A function is a reusable recipe: give it inputs, it gives back an output

def get_rag_status(row):
    # 'row' is one engine's data
    margin = row['EGT_margin_current']
    rul    = row['RUL_current']

    # Check RED conditions first (worst case wins)
    if margin < MARGIN_RED_LIMIT or rul < RUL_RED_LIMIT:
        return 'RED'
    # Then check AMBER
    elif margin < MARGIN_AMBER_LIMIT or rul < RUL_AMBER_LIMIT:
        return 'AMBER'
    # Otherwise it's GREEN
    else:
        return 'GREEN'

# Apply this function to every row in the fleet table
# 'axis=1' means 'apply it row by row' (not column by column)
fleet['RAG_status'] = fleet.apply(get_rag_status, axis=1)

# Count how many engines in each status
print('Fleet RAG breakdown:')
print(fleet['RAG_status'].value_counts())
print()
print('Fleet ready for analysis:')
fleet[['engine_id', 'route_type', 'RAG_status',
       'EGT_margin_current', 'RUL_current',
       'deg_rate_per_100cyc']].head(10)

### What we just built

We now have a **fleet table** with one row per engine, containing:
- Current EGT margin
- Current RUL
- Degradation rate (how fast it's getting worse)
- RAG status (Red / Amber / Green)
- Anomaly count

This is the foundation for every chart we'll build from here.

---

## Q1 — Which engines are behaving abnormally?

We'll answer this with **three charts**, each showing a different angle:

1. **Chart 1:** EGT margin over time for every engine — a bird's eye view
2. **Chart 2:** Anomaly heatmap — which engines had sudden damage events
3. **Chart 3:** Fast degraders — engines losing margin faster than the fleet average

In [ ]:
# Q1 — CHART 1: EGT margin over time for every engine
# 
# We'll draw one line per engine showing how its EGT margin changes over cycles
# The line colour tells us the current RAG status of that engine

# Create a blank chart — 'figsize' is (width, height) in inches
fig, ax = plt.subplots(figsize=(14, 6))

# We need to loop through every engine and draw its line
# 'for' is how we repeat something — here, we repeat once per engine

for engine_id in fleet['engine_id']:

    # Get just this engine's data from the full dataset
    # 'df[df["engine_id"] == engine_id]' means 'rows where engine_id matches'
    engine_data = df[df['engine_id'] == engine_id]

    # Smooth the line using a rolling average over 80 cycles
    # Real data is noisy — smoothing makes the trend easier to see
    smooth_margin = engine_data['EGT_margin'].rolling(80).mean()

    # Find out the RAG status of this engine (look it up in the fleet table)
    status = fleet[fleet['engine_id'] == engine_id]['RAG_status'].values[0]

    # Pick the right colour based on status
    if status == 'RED':
        line_colour = RED
        line_width  = 2.2
    elif status == 'AMBER':
        line_colour = AMBER
        line_width  = 1.5
    else:
        line_colour = GREEN
        line_width  = 1.0

    # Draw the line on the chart
    ax.plot(engine_data['cycle'], smooth_margin,
            color=line_colour, linewidth=line_width, alpha=0.8)

# Add the threshold lines so we can see where red and amber start
ax.axhline(MARGIN_RED_LIMIT,   color=RED,   linestyle='--', label=f'Red line ({MARGIN_RED_LIMIT}°C)')
ax.axhline(MARGIN_AMBER_LIMIT, color=AMBER, linestyle='--', label=f'Amber line ({MARGIN_AMBER_LIMIT}°C)')

# Add titles and labels
ax.set_title('Q1: EGT Margin Trend — All 20 Engines', fontsize=14)
ax.set_xlabel('Flight Cycle')
ax.set_ylabel('EGT Margin (°C)')
ax.legend(loc='upper right')

# Display the chart
plt.tight_layout()
plt.show()

print('Reading this chart:')
print('  • Lines going DOWN = engine is degrading (normal over time)')
print('  • RED lines    = engines needing urgent attention')
print('  • AMBER lines  = engines to watch closely')
print('  • GREEN lines  = healthy engines')

In [ ]:
# Q1 — CHART 2: Anomaly heatmap
# 
# A heatmap is a grid where colour intensity shows a value
# Here: rows = engines, columns = cycle ranges, colour = number of anomalies
# Bright = many anomaly events in that period

# First, group the cycles into 100-cycle buckets to make the chart readable
# Every 100 cycles becomes one column
df['cycle_bucket'] = (df['cycle'] // 100) * 100

# Count anomalies per engine per cycle bucket
# Pivot table = a 2D summary with engines as rows and buckets as columns
anomaly_grid = df[df['is_anomaly'] == 1].pivot_table(
    index   = 'engine_id',        # one row per engine
    columns = 'cycle_bucket',     # one column per cycle bucket
    values  = 'is_anomaly',       # what to count
    aggfunc = 'sum',              # add them up
    fill_value = 0,               # no anomalies = 0 (not blank)
)

# Now draw the heatmap
fig, ax = plt.subplots(figsize=(16, 7))

# seaborn's heatmap does the colouring for us
sns.heatmap(
    anomaly_grid,
    cmap='YlOrRd',            # Yellow → Orange → Red colour scale
    linewidths=0.3,
    ax=ax,
    cbar_kws={'label': 'Anomaly events', 'shrink': 0.6},
)

ax.set_title('Q1: Anomaly Events Heatmap\n(Brighter cells = more sudden damage events)',
             fontsize=13)
ax.set_xlabel('Cycle Range (100-cycle buckets)')
ax.set_ylabel('Engine')
plt.tight_layout()
plt.show()

# Print the top 5 engines with the most anomalies
print('Top 5 engines by anomaly count:')
top_5 = fleet.sort_values('anomaly_count', ascending=False).head(5)
print(top_5[['engine_id', 'route_type', 'anomaly_count', 'RAG_status']].to_string(index=False))

In [ ]:
# Q1 — CHART 3: Fast degraders (scatter plot)
# 
# The idea: an engine can have OK margin now but be degrading very fast
# We want to spot those — the 'rogue' engines
# 
# Plot each engine as a dot:
#   X axis = how fast it's degrading
#   Y axis = current EGT margin
#   Colour = RAG status
#   Engines in the 'bad corner' (fast degradation + low margin) need attention

# Find the fleet median degradation rate — half the engines are above, half below
median_rate = fleet['deg_rate_per_100cyc'].median()

# Flag engines that are degrading 1.5x faster than the median
# '>' means greater than; we create a True/False column
fleet['fast_degrader'] = fleet['deg_rate_per_100cyc'] > (median_rate * 1.5)

# Create the scatter plot
fig, ax = plt.subplots(figsize=(12, 7))

# Loop through every engine and plot one dot
for _, row in fleet.iterrows():

    # Pick colour by RAG status
    if row['RAG_status'] == 'RED':
        dot_colour = RED
    elif row['RAG_status'] == 'AMBER':
        dot_colour = AMBER
    else:
        dot_colour = GREEN

    # Fast degraders get a star shape and bigger size
    if row['fast_degrader']:
        marker_shape = '*'
        dot_size     = 300
    else:
        marker_shape = 'o'
        dot_size     = 120

    # Draw the dot
    ax.scatter(row['deg_rate_per_100cyc'], row['EGT_margin_current'],
               color=dot_colour, marker=marker_shape, s=dot_size,
               alpha=0.8, edgecolors='white', linewidths=0.8)

    # Label each dot with the engine number (last 3 characters of ID)
    label = row['engine_id'][-3:]
    ax.annotate(label,
                (row['deg_rate_per_100cyc'], row['EGT_margin_current']),
                fontsize=8, xytext=(6, 6), textcoords='offset points',
                color='white', alpha=0.8)

# Add threshold lines
ax.axhline(MARGIN_RED_LIMIT,   color=RED,   linestyle='--', alpha=0.6)
ax.axhline(MARGIN_AMBER_LIMIT, color=AMBER, linestyle='--', alpha=0.6)
ax.axvline(median_rate * 1.5,  color='white', linestyle=':', alpha=0.5,
           label=f'Fast degrader threshold ({median_rate*1.5:.2f})')

ax.set_title('Q1: Fast Degraders — Current Margin vs Degradation Rate\n'
             '(Stars = engines degrading >1.5× fleet median)', fontsize=13)
ax.set_xlabel('Degradation Rate (°C margin lost per 100 cycles)')
ax.set_ylabel('Current EGT Margin (°C)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

# List the fast degraders
fast_engines = fleet[fleet['fast_degrader']]
print(f'Found {len(fast_engines)} fast degraders:')
print(fast_engines[['engine_id', 'route_type', 'RAG_status',
                    'EGT_margin_current', 'deg_rate_per_100cyc']].to_string(index=False))

### Q1 — What we learned

- **Chart 1** showed how every engine's margin has trended over its life
- **Chart 2** showed which engines had the most anomaly events
- **Chart 3** identified fast degraders — engines to watch even if they look OK right now

---

## Q2 — RAG Classification Dashboard

Now we'll build the dashboard a maintenance manager would see first thing in the morning.  
Every engine shown with its status at a glance.

In [ ]:
# Q2 — CHART: RAG bar chart showing every engine's current health
# 
# Each bar = one engine, height = current EGT margin, colour = RAG status
# We also put the RUL number above each bar

# Sort engines by margin (worst at left, best at right)
fleet_sorted = fleet.sort_values('EGT_margin_current')

# Build a list of colours — one per engine
bar_colours = []
for status in fleet_sorted['RAG_status']:
    if status == 'RED':
        bar_colours.append(RED)
    elif status == 'AMBER':
        bar_colours.append(AMBER)
    else:
        bar_colours.append(GREEN)

# Create the chart
fig, ax = plt.subplots(figsize=(15, 6))

# Draw the bars
bars = ax.bar(fleet_sorted['engine_id'],
              fleet_sorted['EGT_margin_current'],
              color=bar_colours, edgecolor='#0f1117')

# Add the RUL number above each bar
# 'zip' lets us loop through two lists at the same time
for bar, rul in zip(bars, fleet_sorted['RUL_current']):
    # The x position is the middle of the bar; y is just above the top
    x_pos = bar.get_x() + bar.get_width() / 2
    y_pos = bar.get_height() + 2
    ax.text(x_pos, y_pos, f'{int(rul)}',
            ha='center', fontsize=8, color='white')

# Threshold lines
ax.axhline(MARGIN_RED_LIMIT,   color=RED,   linestyle='--', alpha=0.7)
ax.axhline(MARGIN_AMBER_LIMIT, color=AMBER, linestyle='--', alpha=0.7)

ax.set_title('Q2: Fleet RAG Dashboard\n'
             '(Bar height = current EGT margin | Number above bar = RUL in cycles)',
             fontsize=13)
ax.set_xlabel('Engine')
ax.set_ylabel('Current EGT Margin (°C)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Print the action list
print('MORNING BRIEFING — FLEET STATUS:')
print('─' * 50)

for status_name in ['RED', 'AMBER', 'GREEN']:
    subset = fleet[fleet['RAG_status'] == status_name]
    if len(subset) == 0:
        continue
    print(f'\n{status_name} ({len(subset)} engines):')
    for _, eng in subset.iterrows():
        print(f'  {eng["engine_id"]}  '
              f'Margin: {eng["EGT_margin_current"]:.1f}°C  '
              f'RUL: {eng["RUL_current"]} cycles')

---
## Q3 — RUL Projection for Red & Amber Engines

For engines that need maintenance soon, we want to project when exactly they'll hit the minimum margin.

**Approach:** Fit a straight trend line through the historical data and extend it forward into the future.  
This tells us the predicted crossing point — when the margin will hit the red threshold.

In [ ]:
# Q3 — Get the engines that need maintenance (red or amber)
# Then draw a projection chart for each one

# Filter to just the concerning engines
needs_maintenance = fleet[fleet['RAG_status'].isin(['RED', 'AMBER'])]
needs_maintenance = needs_maintenance.sort_values('EGT_margin_current')

print(f'Engines needing projection: {len(needs_maintenance)}')
print(needs_maintenance[['engine_id', 'RAG_status', 'EGT_margin_current', 'RUL_current']].to_string(index=False))

In [ ]:
# Q3 — Draw projection charts
# 
# For each concerning engine:
# 1. Plot its historical margin over cycles
# 2. Fit a straight line through the history
# 3. Extend that line into the future
# 4. Mark where it crosses the red threshold

# Figure out grid size — 3 charts per row
num_engines = len(needs_maintenance)
num_cols = 3
num_rows = (num_engines + num_cols - 1) // num_cols   # round up

fig, axes = plt.subplots(num_rows, num_cols,
                         figsize=(5 * num_cols, 4 * num_rows))

# If only one row, put it in a list so indexing works the same way
if num_rows == 1:
    axes = [axes]

# Flatten the grid of axes into a single list
axes_flat = []
for row in axes:
    if hasattr(row, '__len__'):
        for ax in row:
            axes_flat.append(ax)
    else:
        axes_flat.append(row)

# Loop through each engine and draw its projection
engine_list = list(needs_maintenance['engine_id'])

for i, engine_id in enumerate(engine_list):

    ax = axes_flat[i]

    # Get this engine's full history
    engine_data = df[df['engine_id'] == engine_id].copy()

    # Smooth with a rolling average
    engine_data['margin_smooth'] = engine_data['EGT_margin'].rolling(80).mean()

    # Drop rows where smoothing produced no value
    engine_data_clean = engine_data.dropna(subset=['margin_smooth'])

    # Get x and y for the trend fit
    cycles   = engine_data_clean['cycle'].values
    margins  = engine_data_clean['margin_smooth'].values

    # Fit a straight line — numpy's polyfit finds the best line through the points
    # Degree 1 means straight line (y = slope * x + intercept)
    slope, intercept = np.polyfit(cycles, margins, 1)

    # Calculate future cycles we want to project into
    last_cycle    = cycles[-1]
    future_end    = last_cycle + 1500
    future_cycles = np.arange(last_cycle, future_end)

    # Apply the line equation to get projected margins
    projected_margins = slope * future_cycles + intercept

    # Find where the projection crosses the red threshold
    # This gives us the 'when will it need the shop' answer
    crosses = projected_margins <= MARGIN_RED_LIMIT
    if crosses.any():
        crossing_idx   = np.where(crosses)[0][0]
        crossing_cycle = future_cycles[crossing_idx]
    else:
        crossing_cycle = None

    # Get RAG colour
    rag_status = needs_maintenance[needs_maintenance['engine_id'] == engine_id]['RAG_status'].values[0]
    rag_col = RED if rag_status == 'RED' else AMBER

    # Plot raw data as light scatter
    ax.scatter(engine_data['cycle'], engine_data['EGT_margin'],
               alpha=0.15, s=3, color=rag_col)

    # Plot smoothed history
    ax.plot(cycles, margins, color=rag_col, linewidth=2, label='Observed trend')

    # Plot the projection
    ax.plot(future_cycles, projected_margins,
            color='white', linestyle='--', linewidth=1.5,
            alpha=0.7, label='Projected')

    # Add threshold line
    ax.axhline(MARGIN_RED_LIMIT, color=RED, linestyle=':', alpha=0.6)

    # Mark the crossing
    if crossing_cycle is not None:
        ax.axvline(crossing_cycle, color=RED, linestyle=':', alpha=0.8)
        ax.annotate(f'~{crossing_cycle:.0f}',
                    (crossing_cycle, MARGIN_RED_LIMIT),
                    xytext=(5, 10), textcoords='offset points',
                    fontsize=9, color=RED)

    rul = needs_maintenance[needs_maintenance['engine_id'] == engine_id]['RUL_current'].values[0]
    ax.set_title(f'{engine_id} | {rag_status} | RUL: {rul}', fontsize=10)
    ax.set_xlabel('Cycle')
    ax.set_ylabel('EGT Margin (°C)')
    ax.legend(fontsize=7)

# Hide empty subplots if there are more grid spots than engines
for j in range(num_engines, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Q3: RUL Projection — When Will Each Engine Need the Shop?',
             fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

print('Reading these charts:')
print('  • Solid line = observed trend from real data')
print('  • Dashed line = our projection into the future')
print('  • Number at crossing = predicted cycle when engine hits red threshold')

---
## Q4 — Sensor Health Audit

Are the sensors giving us reliable data?  
We check two things: **missing data rate** (sensor dropouts) and **outlier rate** (spike readings).

In [ ]:
# Q4 — CHART 1: Sensor dropout rates
# 
# How often does each sensor produce a missing reading?
# Below 1% = good, 1-3% = watch, above 3% = sensor needs inspection

# List of sensor columns we want to check
sensor_columns = ['vibration_1', 'vibration_2', 'oil_consumption',
                  'EGT', 'N1', 'N2', 'fuel_flow', 'oil_temp', 'oil_pressure']

# For each sensor, calculate what percentage of rows are missing
# '.isna()' returns True where missing, '.mean() * 100' gives us the percentage
dropout_rates = {}
for sensor in sensor_columns:
    pct_missing = df[sensor].isna().mean() * 100
    dropout_rates[sensor] = pct_missing

# Convert to a pandas Series so it's easy to plot
dropout_series = pd.Series(dropout_rates).sort_values()

# Pick colours: green if <1%, amber if 1-3%, red if >3%
dropout_colours = []
for pct in dropout_series:
    if pct > 3:
        dropout_colours.append(RED)
    elif pct > 1:
        dropout_colours.append(AMBER)
    else:
        dropout_colours.append(GREEN)

# Draw the chart
fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(dropout_series.index, dropout_series.values,
        color=dropout_colours, edgecolor='#0f1117')

# Threshold lines
ax.axvline(1, color=AMBER, linestyle='--', alpha=0.7, label='1% (watch)')
ax.axvline(3, color=RED,   linestyle='--', alpha=0.7, label='3% (action)')

ax.set_title('Q4: Sensor Dropout Rates (Fleet-Wide)', fontsize=13)
ax.set_xlabel('% of Readings Missing')
ax.legend()
plt.tight_layout()
plt.show()

print('Sensor dropout summary:')
for sensor, pct in dropout_series.items():
    print(f'  {sensor:<18}: {pct:.2f}%')

In [ ]:
# Q4 — CHART 2: Outlier detection per engine
# 
# An outlier is a reading that's very different from the engine's own recent average
# If one engine has many outliers, its sensor might be faulty
# 
# We use a simple rule: if a reading is more than 3x the typical variation away
# from the rolling average, it's an outlier

# We'll check EGT outliers per engine
# The approach: for each engine, compare each reading to its own rolling stats

outlier_counts = {}

for engine_id in fleet['engine_id']:

    # Get this engine's EGT values
    engine_egt = df[df['engine_id'] == engine_id]['EGT']

    # Rolling average over 50 cycles
    rolling_avg = engine_egt.rolling(50).mean()

    # Rolling standard deviation over 50 cycles (measures spread)
    rolling_std = engine_egt.rolling(50).std()

    # How many standard deviations away is each point?
    # Adding a tiny 0.0001 prevents division by zero errors
    z_scores = (engine_egt - rolling_avg).abs() / (rolling_std + 0.0001)

    # Count points that are more than 3 std away — these are outliers
    num_outliers = (z_scores > 3).sum()

    # Store as percentage
    outlier_counts[engine_id] = (num_outliers / len(engine_egt)) * 100

outlier_series = pd.Series(outlier_counts).sort_values()

# Colour by level
outlier_colours = []
for pct in outlier_series:
    if pct > 1:
        outlier_colours.append(RED)
    elif pct > 0.5:
        outlier_colours.append(AMBER)
    else:
        outlier_colours.append(GREEN)

# Draw
fig, ax = plt.subplots(figsize=(11, 7))
# Show just the engine number (last 3 chars) for cleaner labels
engine_labels = [e[-3:] for e in outlier_series.index]
ax.barh(engine_labels, outlier_series.values,
        color=outlier_colours, edgecolor='#0f1117')

ax.axvline(0.5, color=AMBER, linestyle='--', alpha=0.7, label='0.5% (watch)')
ax.axvline(1.0, color=RED,   linestyle='--', alpha=0.7, label='1% (suspect)')

ax.set_title('Q4: EGT Outlier Rate Per Engine\n'
             '(How often is EGT very different from rolling average?)', fontsize=12)
ax.set_xlabel('Outlier % (readings >3 standard deviations from rolling mean)')
ax.set_ylabel('Engine')
ax.legend()
plt.tight_layout()
plt.show()

# Call out suspicious engines
suspects = outlier_series[outlier_series > 1]
if len(suspects) > 0:
    print(f'Engines with elevated outlier rates (>1%):')
    for eng, pct in suspects.items():
        print(f'  {eng}: {pct:.2f}%')
else:
    print('No engines with suspicious outlier rates.')

---
## Q5 — Route Type vs Engine Life Consumption

Do short-haul engines degrade faster than long-haul ones?  
Short-haul = more takeoffs/landings per hour → more thermal cycling → faster wear.

In [ ]:
# Q5 — CHART 1: Box plot of degradation rate by route
# 
# A box plot shows:
#   • The box = where the middle 50% of engines sit
#   • The line in the box = the median (typical engine)
#   • The whiskers = the range
# 
# If short-haul box sits higher than long-haul box, short-haul is degrading faster

# Colours for each route type
route_colours_map = {
    'short_haul':  RED,      # red = fastest degradation (our hypothesis)
    'medium_haul': AMBER,
    'long_haul':   GREEN,
}

fig, ax = plt.subplots(figsize=(10, 6))

# Prepare data — one list of rates per route type
routes_in_order = ['short_haul', 'medium_haul', 'long_haul']
box_data = []
for route in routes_in_order:
    rates = fleet[fleet['route_type'] == route]['deg_rate_per_100cyc'].values
    box_data.append(rates)

# Draw the box plot
bp = ax.boxplot(box_data,
                labels=['Short Haul', 'Medium Haul', 'Long Haul'],
                patch_artist=True,    # allows us to colour the boxes
                widths=0.5)

# Apply our colours to each box
for patch, route in zip(bp['boxes'], routes_in_order):
    patch.set_facecolor(route_colours_map[route])
    patch.set_alpha(0.7)

# Make the median lines white so they stand out
for median_line in bp['medians']:
    median_line.set_color('white')
    median_line.set_linewidth(2)

ax.set_title('Q5: Degradation Rate by Route Type', fontsize=13)
ax.set_ylabel('EGT Margin Lost per 100 Cycles (°C)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Print the actual numbers
print('Average degradation rate by route:')
for route in routes_in_order:
    avg = fleet[fleet['route_type'] == route]['deg_rate_per_100cyc'].mean()
    print(f'  {route:<15}: {avg:.3f} °C per 100 cycles')

In [ ]:
# Q5 — CHART 2: Average EGT margin trajectory by route type
# 
# Draw one smooth average line per route, showing how the typical
# engine on that route degrades over time

fig, ax = plt.subplots(figsize=(13, 6))

# For each route type, calculate the average margin at each cycle position
for route in routes_in_order:

    # Get engines on this route
    route_engines = fleet[fleet['route_type'] == route]['engine_id'].tolist()
    route_data    = df[df['engine_id'].isin(route_engines)].copy()

    # Normalise cycle — each engine starts at 0 so we can compare fairly
    # (one engine might start at cycle 100, another at cycle 500 due to initial age)
    route_data['cycle_norm'] = route_data.groupby('engine_id')['cycle'].transform(
        lambda x: x - x.min()
    )

    # Average margin at each normalised cycle position
    avg_by_cycle = route_data.groupby('cycle_norm')['EGT_margin'].mean()

    # Smooth it
    smooth_avg = avg_by_cycle.rolling(100).mean()

    # Plot
    ax.plot(smooth_avg.index, smooth_avg.values,
            color=route_colours_map[route], linewidth=2.5,
            label=f'{route.replace("_", " ").title()} (n={len(route_engines)})')

# Thresholds
ax.axhline(MARGIN_RED_LIMIT,   color=RED,   linestyle='--', alpha=0.5)
ax.axhline(MARGIN_AMBER_LIMIT, color=AMBER, linestyle='--', alpha=0.5)

ax.set_title('Q5: Average EGT Margin Over Life — By Route Type', fontsize=13)
ax.set_xlabel('Cycles Since Engine Entered Service')
ax.set_ylabel('Average EGT Margin (°C)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('Reading this chart:')
print('  • A steeper line = faster degradation')
print('  • If short-haul is steepest, the hypothesis holds')

---
## Phase 2 Complete

### What you built:

| Question | Answer |
|---|---|
| Q1 — Abnormal engines | Identified via trend chart, anomaly heatmap, and fast-degrader scatter |
| Q2 — RAG classification | One-glance dashboard with RUL overlays |
| Q3 — How long do they have? | Linear projection with predicted redline crossing cycles |
| Q4 — Sensor audit | Dropout rates and outlier detection per engine |
| Q5 — Route comparison | Boxplots and trajectory lines per route type |

### What you should practise next:
- Re-run the notebook with different threshold values (change `MARGIN_RED_LIMIT` to 25 and see what happens)
- Try sorting the fleet differently — by RUL instead of margin
- Read the Phase 2 output numbers and write a 3-sentence story about your fleet

### Key Python concepts you've now used:
- **DataFrame** (`df`) — a table of data
- **`groupby`** — split data into groups and summarise each
- **`for` loop** — repeat something for each item in a list
- **`if/elif/else`** — branching logic
- **Function** (`def ...`) — a reusable recipe
- **Plot** (`ax.plot`, `ax.bar`) — drawing charts

### Next up: Phase 3 — Anomaly Detection

---